# FCSG-Net training on Colab

Overflow path, reserved for the Phase 4 ablations so the Kaggle quota goes to
the main runs. Colab storage is ephemeral, so checkpoints go to Drive or they
are gone when the runtime recycles.

Runtime -> Change runtime type -> T4 GPU, first.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/fcsg-net/ckpt"   # survives the runtime dying
!mkdir -p "$OUT"

In [ ]:
import os, subprocess
REPO = "/content/fcsg-capstone"
if os.path.isdir(REPO):
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/whynotramaa/fcsg-capstone", REPO], check=True)
os.chdir(REPO)

## Data

No attached datasets here, so fetch DIV2K over the network. About 3.3 GB and a
few minutes on Colab's link. Re-runs skip the download if the directory is
already there.

In [ ]:
%%bash
mkdir -p /content/DIV2K && cd /content/DIV2K
for split in train valid; do
  d=DIV2K_${split}_HR
  [ -d "$d" ] && { echo "$d present"; continue; }
  wget -q --show-progress http://data.vision.ee.ethz.ch/cvl/DIV2K/${d}.zip
  unzip -q ${d}.zip && rm ${d}.zip
done
ls -d /content/DIV2K/*/

In [ ]:
!python training/train.py --config configs/dncnn.toml --data /content/DIV2K --out "$OUT"

In [ ]:
import glob
ckpt = sorted(glob.glob(OUT + "/ckpt_*.pt"))[-1]
!python evaluation/eval.py --ckpt $ckpt --data /content/DIV2K --out /content/results --limit 20
!python evaluation/plots.py --csv "$OUT/train_log.csv" --out /content/results/figures